In [195]:
import nest_asyncio
nest_asyncio.apply()
from google.oauth2.credentials import Credentials
from google.auth.transport.requests import Request
from google_auth_oauthlib.flow import InstalledAppFlow
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import RunReportRequest, DateRange, Dimension, Metric
import pandas as pd
from datetime import timedelta,datetime
import os
import numpy as np 
from email.message import EmailMessage
import sys
from scipy import stats

In [196]:
PROPERTY_ID = "properties/220996928"  # replace with your actual GA4 property ID
SCOPES = ["https://www.googleapis.com/auth/analytics.readonly"]

In [197]:
# command for assigning token 
def cred_save():
    cred=None
    if os.path.exists("token.json"):
        cred=Credentials.from_authorized_user_file("token.json",SCOPES)
    if not cred or not cred.valid:
        if cred and cred.expired and cred.refresh_token:
            cred.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file("client_secret_luvkey.json", SCOPES)
            cred=flow.run_local_server(port=0)
        with open("token.json","w") as token:
            token.write(cred.to_json())
    return cred
    

In [198]:
credentials=cred_save()
client= BetaAnalyticsDataClient(credentials=credentials)

In [199]:
# creating the date  variables 
today=datetime.now().date()
yesterday=today-timedelta(days=1)
start=yesterday-timedelta(days=30)

In [200]:
#  Active Users for yesterday 

request = RunReportRequest(
    property=PROPERTY_ID,
    dimensions=[Dimension(name="date")],
    metrics=[Metric(name="activeUsers")],
    date_ranges=[DateRange(start_date=str(start), end_date=str(yesterday))],
)
response = client.run_report(request)

In [201]:
"""# --- PRINT RESULT ---
print("Raw response rows:")

for row in response.rows:
    date = row.dimension_values[0].value
    active_users = row.metric_values[0].value
    print(f"Date: {date} | Active Users: {active_users}")


if not response.rows:
    print("No data returned — check property ID, date range, or permissions.")
"""

'# --- PRINT RESULT ---\nprint("Raw response rows:")\n\nfor row in response.rows:\n    date = row.dimension_values[0].value\n    active_users = row.metric_values[0].value\n    print(f"Date: {date} | Active Users: {active_users}")\n\n\nif not response.rows:\n    print("No data returned — check property ID, date range, or permissions.")\n'

In [202]:
d=[]

for row in response.rows:
    dimension_val=row.dimension_values[0].value
    metric_val=row.metric_values[0].value
    d.append({"date":dimension_val,"average user daily":metric_val})



In [203]:
df=pd.DataFrame(d)

In [204]:
df['date']=pd.to_datetime(df['date'],format="%Y%m%d")

In [205]:

df=df.sort_values(by='date',ascending=True)
df.reset_index(drop=True)


,date,average user daily
0,2026-05-25,69914
1,2026-05-26,68754
2,2026-05-27,70323
3,2026-05-28,68239
4,2026-05-29,69309
5,2026-05-30,72361
6,2026-05-31,75177
7,2026-06-01,72371
8,2026-06-02,72423
9,2026-06-03,73553


In [ ]:
df['average user daily']=df['average user daily'].astype(int)
df1=df[:30].copy()

In [ ]:
from scipy import stats
h=df1['average user daily'].values
skew_val=stats.skew(h)
skew_val

np.float64(-1.2651625494140537)

In [ ]:

y=df1["average user daily"].values
if skew_val>0.5 or skew_val<-0.5:
    q1 = np.percentile(y, 25)
    q3 = np.percentile(y, 75)
    iqr = q3 - q1
    upper = q3 + 1.5 * iqr   # or use np.percentile(y, 90) for a simpler read
    lower = q1 - 1.5 * iqr
else:
    standard_dev=np.std(y)
    mean=np.mean(y)
    upper=mean+standard_dev
    lower=mean-2*standard_dev
yesterday_val=df[30:]

In [209]:
yesterday_val=yesterday_val["average user daily"].values[0].item()

In [210]:
# I will be using asymmetric anamoly detection  
import json
import smtplib


In [211]:
with open("email_credential.json","r") as file:
    data=json.load(file)
semail_id=data["email"]
semail_password=data["password"]
if not semail_id or semail_password:
    ValueError("credential not found")



In [212]:
msg=EmailMessage()

receiver_email="66prakhar676@gmail.com"

msg["Subject"] = "Test Email from my Python Script! 🚀"
msg["From"] = semail_id
msg["To"] = receiver_email
if yesterday_val>upper:
        msg.set_content("it is a good day")
elif yesterday_val<lower:
        msg.set_content("DAU down alert")
else :
        print("normal day")
        sys.exit()

In [213]:
try:
  server=smtplib.SMTP("smtp.gmail.com",587)
  server.starttls()
  server.login(semail_id,semail_password)

  print("server logged successfully")
  server.send_message(msg)
  print("message sent successfully")
except smtplib.SMTPAuthenticationError:
   print("Login Failed check your email or password")
except Exception as e:
   print(f"network error as :{e}")
finally:
   if server in locals():
      server.quit()


server logged successfully
message sent successfully
